# Weather Underground PWS Data Pipeline

Fetch historical weather data from Weather Underground Personal Weather Stations (PWS) via API.

**Source:** https://api.weather.com/v2/pws

## Key Features
- Historical hourly data (aggregated from minute-level observations)
- Clean column names with precipitation data prioritized
- Automatic chunking for date ranges > 31 days
- Quality checks and validation
- Multi-station support

## Pipeline Steps
1. Configure API key and stations
2. Fetch historical data (auto-chunked if needed)
3. Convert to clean DataFrames with standardized column names
4. Quality checks and data validation
5. Analyze precipitation patterns
6. Plot and save results

## 1. Configuration

Configure API key, stations, date range, and output directory.

**Stations:** Use PWS station IDs (e.g., 'KNYNEWYO1805', 'KNYNEWYO1850')

**Date Range:** Start and end dates (datetime objects)

**Output Directory:** Where processed CSV files will be saved

**API Key:** Set via environment variable `WU_API_KEY` or in config.py

In [ ]:
# Setup: Import libraries and configure pipeline
from datetime import datetime
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Import from local files
from config import get_api_key, check_api_key_status
from wu_fetch import *
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# CONFIGURATION - EDIT THESE VALUES
# ============================================================
STATION_IDS = ['KNYNEWYO1805', 'KNYNEWYO1850']  # Your PWS station IDs
START_DATE  = datetime(2024, 1, 1)
END_DATE    = datetime(2024, 1, 30)

# Output directory (relative to this notebook)
OUTPUT_DIR = Path.cwd() / 'output' / 'wu'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

# Check API key status
api_status = check_api_key_status()
print(api_status['message'])

if api_status['available']:
    API_KEY = get_api_key()
else:
    API_KEY = None
    print("\nOptions to set API key:")
    print("  1. Environment variable: export WU_API_KEY='your_key'")
    print("  2. Add to config.py: WU_API_KEY = 'your_key'")
    print("See API_KEY_SETUP.md for detailed instructions.")

print(f'\nStations: {STATION_IDS}')
print(f'Period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')
print('\n✓ Setup complete')

## 2. Fetch Raw Data

Fetch historical weather data from Weather Underground API.

**Note:** Data is fetched in chunks if the date range exceeds 31 days (API limit). The pipeline automatically handles chunking and merging.

**Data Resolution:** Hourly (aggregated from minute-level PWS observations)

**API Key Required:** Weather Underground requires an API key for authentication.

### 2.0 API Test (Optional - Can Skip)

This cell tests the WU API to verify it's working correctly. You can skip this cell if you trust the API is functioning.

In [ ]:
# Optional: Test API connection with a single station, single day
if API_KEY:
    print("Testing API connection...")
    test_result = get_historical_data_chunk(
        api_key=API_KEY,
        station_id=STATION_IDS[0],
        start_date=START_DATE,
        end_date=START_DATE + timedelta(days=1),
        units='m',
        data_type='hourly'
    )
    
    if test_result and 'observations' in test_result:
        print(f"✓ API test successful! Got {len(test_result['observations'])} observations")
        print(f"  Sample observation keys: {list(test_result['observations'][0].keys())[:5]}...")
    else:
        print("✗ API test failed or no data returned")
else:
    print("✗ API key not set - cannot test")

### 2.1 Fetch Data

Fetch historical data for all configured stations.

In [ ]:
# Fetch data using the pipeline
if not API_KEY:
    print("❌ API key not found. Cannot fetch data.")
    results = None
else:
    results = run_wu_pipeline(
        api_key=API_KEY,
        station_ids=STATION_IDS,
        start_date=START_DATE,
        end_date=END_DATE,
        units='m',  # 'm' for metric, 'e' for imperial
        fetch_options={'historical': True},
        output_dir=None,  # We'll save manually later
        save_data=False
    )

### 2.2 Quick Look at Raw Data

Inspect the raw data structure: columns, shape, and date ranges.

In [ ]:
# Quick look at fetched data
if results and 'dataframes' in results:
    all_dfs = results['dataframes']
    
    for station_id, dfs in all_dfs.items():
        if 'clean' in dfs:
            df = dfs['clean']
            print(f"\n{station_id}:")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)[:10]}...")
            
            # Find datetime column
            dt_col = None
            for col in ['datetime', 'time_local', 'obsTimeLocal']:
                if col in df.columns:
                    dt_col = col
                    break
            
            if dt_col:
                print(f"  Date range: {df[dt_col].min()} to {df[dt_col].max()}")
else:
    print("⚠ No data fetched")

## 3. Process and Validate Data

Extract processed (clean) DataFrames and perform quality checks.

**Clean Data:** Standardized column names with precipitation data prioritized

**Quality Checks:** Validate date ranges, check for missing data, verify precipitation values

In [ ]:
# Extract processed data (clean DataFrames)
processed_data = {}

if results and 'dataframes' in results:
    processed_data = {
        sid: dfs['clean'] 
        for sid, dfs in results['dataframes'].items() 
        if 'clean' in dfs
    }
    
    print("\n" + "=" * 60)
    print("PROCESSING RESULTS")
    print("=" * 60)
    print(f"\n✓ Processed {len(processed_data)} station(s)")
    
    # Summary for each station
    for station_id, df in processed_data.items():
        print(f"\n  {station_id}:")
        print(f"    Rows: {len(df):,}")
        
        # Precipitation
        precip_col = next((c for c in ['precip_rate', 'precip_amount', 'precipitation'] if c in df.columns), None)
        if precip_col:
            total_precip = df[precip_col].sum()
            rainy_hours = (df[precip_col] > 0).sum()
            print(f"    Total precip: {total_precip:.1f} mm ({rainy_hours} rainy hours)")
        
        # Temperature
        if 'temperature' in df.columns:
            print(f"    Temp range: {df['temperature'].min():.1f}°C to {df['temperature'].max():.1f}°C")
else:
    print("⚠ No processed data available")

### 3.1 Data Quality Check

In [ ]:
# Quality checks
if processed_data:
    print("\n" + "=" * 60)
    print("DATA QUALITY CHECKS")
    print("=" * 60)
    
    for station_id, df in processed_data.items():
        print(f"\n{station_id}:")
        check_data_quality(df, START_DATE, END_DATE, verbose=True)

## 4. Save Data

Save processed data to CSV files.

**Filename Format:** `WU_YYYY-MM-DD_YYYY-MM-DD.csv`

In [ ]:
# Save processed data
if processed_data:
    save_wu(
        processed_data=processed_data,
        output_dir=OUTPUT_DIR,
        overwrite=True
    )
else:
    print("⚠ No data to save")

## 5. Visualization

Plot precipitation and weather data to visualize patterns and compare stations.

**Available plots:**
- Precipitation analysis (rate and cumulative)
- Multi-station comparison
- Weather dashboard (temperature, humidity, wind, pressure)

### 5.1 Precipitation Analysis

In [ ]:
# Plot precipitation analysis for all stations
if processed_data:
    # Wrap DataFrames in expected format: {station_id: {'clean': df}}
    station_data_for_plot = {sid: {'clean': df} for sid, df in processed_data.items()}
    
    plot_precipitation_analysis_multi(
        station_data=station_data_for_plot,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

### 5.2 Cumulative Precipitation

In [ ]:
# Plot cumulative precipitation
if processed_data:
    station_data_for_plot = {sid: {'clean': df} for sid, df in processed_data.items()}
    
    plot_cumulative_precipitation_multi(
        station_data=station_data_for_plot,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

### 5.3 Weather Dashboard (Single Station)

In [ ]:
# Plot weather dashboard for first station
if processed_data:
    first_station = list(processed_data.keys())[0]
    df_first = processed_data[first_station]
    
    plot_weather_dashboard(
        df=df_first,
        station_id=first_station,
        units='m',
        show=True
    )
else:
    print("⚠ No data to plot")

## 6. Summary

Final summary of the pipeline execution.

In [ ]:
# Summary
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nPeriod: {START_DATE.date()} to {END_DATE.date()}")
print(f"Stations: {len(processed_data)}")

if processed_data:
    # Show first few columns
    sample_cols = list(list(processed_data.values())[0].columns)[:8]
    print(f"\nVariables: {', '.join(sample_cols)}...")
    
    print(f"\nPrecipitation totals:")
    for station_id, df in processed_data.items():
        precip_col = next((c for c in ['precip_rate', 'precip_amount', 'precipitation'] if c in df.columns), None)
        if precip_col:
            total = df[precip_col].sum()
            print(f"  {station_id}: {total:.1f} mm")
        else:
            print(f"  {station_id}: No precipitation data")

print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")
print("\n" + "=" * 60)
print("✓ COMPLETE")

## 7. Load Data (Optional)

Load previously saved data instead of fetching/processing again.

**Use this section if:**
- You've already run the fetch/preprocess steps
- You want to skip to visualization
- You're working with saved data from a previous run

In [ ]:
# Load saved data
LOAD_DATA = False  # Set to True to load instead of fetching

if LOAD_DATA:
    csv_files = list(OUTPUT_DIR.glob('WU_*.csv'))
    
    if csv_files:
        print("=" * 70)
        print("LOADING SAVED DATA")
        print("=" * 70)
        
        # Load most recent file
        latest_file = max(csv_files, key=lambda p: p.stat().st_mtime)
        print(f"\nLoading: {latest_file.name}")
        
        df_loaded = pd.read_csv(latest_file)
        
        # Convert datetime column if present
        for dt_col in ['datetime', 'time_local', 'obsTimeLocal']:
            if dt_col in df_loaded.columns:
                df_loaded[dt_col] = pd.to_datetime(df_loaded[dt_col])
                break
        
        print(f"✓ Loaded {len(df_loaded):,} rows")
        print(f"  Columns: {list(df_loaded.columns)[:10]}...")
        
        # Group by station if station_id column exists
        if 'station_id' in df_loaded.columns:
            processed_data = {sid: group.copy() for sid, group in df_loaded.groupby('station_id')}
            print(f"  Stations: {list(processed_data.keys())}")
        else:
            processed_data = {'all': df_loaded}
            print("  Single station or no station_id column")
    else:
        print(f"✗ No CSV files found in {OUTPUT_DIR.resolve()}")
else:
    print("ℹ Skipping load (set LOAD_DATA=True to load saved data)")